# Demo: probabilistic boson addition

This notebook builds two-parameter photon-number grids for probabilistic boson
addition and analyzes them with the same `analyze_2d` pipeline used by the
Jaynes-Cummings notebook.

The first example uses the closed-form one-boson-added probability model. The
second example shows how to reverse the y-scan direction. The third example
uses the optional QuTiP-backed coherent-state construction for two boson
additions. The final section shows a CSV round trip: how to save a
2D probability grid in a format that could also come from experiment, then
read it back into `Pall` for a fast scan.

In [ ]:
# Make the project importable when running from the notebooks/ directory.
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.config import apply_thread_limits
apply_thread_limits()

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.analytical.models import test_P_pba
from src.comparison.scan_2d import analyze_2d, load_scan_result

try:
    import qutip  # noqa: F401
    from src.analytical.models import pba_Pk_co
    HAS_QUTIP = True
except ImportError:
    HAS_QUTIP = False

print('QuTiP available:', HAS_QUTIP)


## Helpers

In [ ]:
def make_pba_closed_grid(mu_values, p_values, indx):
    """Build Pall[i_p, i_mu, j] from the closed-form one-addition model."""
    Pall = np.empty((len(p_values), len(mu_values), len(indx)))
    for ip, p in enumerate(p_values):
        for imu, mu in enumerate(mu_values):
            for j, k in enumerate(indx):
                Pall[ip, imu, j] = test_P_pba(mu, p, 1, int(k))
    return Pall


def make_pba_qutip_grid(mu_values, p_values, indx, *, n_add, ns=100):
    """Build Pall[i_p, i_mu, j] with the optional QuTiP coherent-state model."""
    if not HAS_QUTIP:
        raise RuntimeError('QuTiP is not installed. Install with: pip install .[physics]')

    Pall = np.empty((len(p_values), len(mu_values), len(indx)))
    for ip, p in enumerate(p_values):
        for imu, mu in enumerate(mu_values):
            for j, k in enumerate(indx):
                Pall[ip, imu, j] = pba_Pk_co(ns, mu, p, int(k), nad=n_add)
    return Pall


def load_scan_archive(path):
    """Read a saved analyze_2d archive back into its components."""
    return load_scan_result(path)


def draw_boundary_curves(boundaries, *, color, label, mode='scatter', lw=2, s=4, min_points=5, max_jump=2.0):
    first = True
    for b in boundaries:
        if len(b) < min_points:
            continue

        if mode == 'scatter':
            plt.scatter(b[:, 0], b[:, 1], s=s, color=color, label=label if first else None)
        elif mode == 'line':
            plt.plot(b[:, 0], b[:, 1], lw=lw, color=color, label=label if first else None)
        elif mode == 'cut-line':
            jumps = np.sqrt(np.sum(np.diff(b, axis=0)**2, axis=1))
            cuts = np.where(jumps > max_jump)[0] + 1
            for piece in np.split(b, cuts):
                if len(piece) < min_points:
                    continue
                plt.plot(piece[:, 0], piece[:, 1], lw=lw, color=color, label=label if first else None)
                first = False
            continue
        else:
            raise ValueError("mode must be 'scatter', 'line', or 'cut-line'")

        first = False


def plot_boundaries(boundaries, boundaries_anaK, boundaries_anaM, title, mu_values, p_values, mode='scatter'):
    plt.figure(figsize=(6, 5))
    draw_boundary_curves(boundaries, color='black', label='LP', mode=mode, lw=4, s=5)
    draw_boundary_curves(boundaries_anaK, color='red', label='analytical K', mode=mode, lw=3, s=4)
    draw_boundary_curves(boundaries_anaM, color='blue', label='analytical M', mode=mode, lw=3, s=4)

    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    if by_label:
        plt.legend(by_label.values(), by_label.keys())
    plt.xlim(0, float(np.max(mu_values)))
    plt.ylim(0, float(np.max(p_values)))
    plt.xlabel(r'$\mu$')
    plt.ylabel(r'$p$')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def classification_map(data):
    """0 classical, 1 analytical-only, 2 LP-only, 3 both, NaN non-physical."""
    num = data['nonclass_num']
    ana = data['nonclass_ana']
    valid = np.isfinite(num) & np.isfinite(ana)
    classes = np.full(num.shape, np.nan, dtype=float)
    classes[valid] = 0.0
    classes[valid & (ana > 0.5)] = 1.0
    classes[valid & (num > 0.5)] = 2.0
    classes[valid & (num > 0.5) & (ana > 0.5)] = 3.0
    return classes


def map_cmap():
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad('white')
    return cmap


def plot_map(data, title, boundary_mode='scatter'):
    classes = classification_map(data)
    plt.figure(figsize=(6, 5))
    plt.pcolormesh(
        data['x'], data['y'],
        np.ma.masked_invalid(classes),
        shading='auto', cmap=map_cmap(),
    )
    plt.colorbar(label='0 classical / 1 analytical-only / 2 LP-only / 3 both')
    draw_boundary_curves(data['boundaries'], color='red', label='LP boundary', mode=boundary_mode, lw=2, s=3)
    plt.xlim(0, float(np.max(data['x'])))
    plt.ylim(0, float(np.max(data['y'])))
    plt.xlabel(r'$\mu$')
    plt.ylabel(r'$p$')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def detection_summary(data, label):
    """Print how much the LP and analytical maps agree on physical points."""
    num = data['nonclass_num']
    ana = data['nonclass_ana']
    valid = np.isfinite(num) & np.isfinite(ana)
    num_detected = valid & (num > 0.5)
    ana_detected = valid & (ana > 0.5)
    missed = ana_detected & ~num_detected
    num_only = num_detected & ~ana_detected
    invalid = np.size(valid) - int(np.count_nonzero(valid))
    print(
        f"{label}: LP={num_detected.sum()} | analytical={ana_detected.sum()} | "
        f"analytical-only={missed.sum()} | LP-only={num_only.sum()} | non-physical={invalid}"
    )


def probability_scale_report(Pall, mu_values, p_values, *, threshold=1e-12):
    """Report where the selected probabilities are very small."""
    selected_mass = np.sum(Pall, axis=2)
    small = selected_mass < threshold
    print('min sum over selected probabilities:', np.min(selected_mass))
    print(f'grid points with selected mass < {threshold:g}:', int(np.sum(small)), 'of', selected_mass.size)
    if np.any(small):
        coords = np.argwhere(small)
        ip, imu = coords[0]
        print('first small point: p =', p_values[ip], '| mu =', mu_values[imu], '| selected mass =', selected_mass[ip, imu])


def run_pba_case(case, mu_values, p_values, Pall):
    os.makedirs(os.path.dirname(os.path.abspath(case['fi'])), exist_ok=True)
    solver_kwargs = case.get('solver_kwargs', globals().get('lp_precision_kwargs', {}))
    result = analyze_2d(
        Pall,
        x_values=mu_values,
        y_values=p_values,
        indx=case['indx'],
        fi=case['fi'],
        fixed_values=np.array([case['n_add']]),
        use_ana_m=case.get('use_ana_m', True),
        use_cm=case.get('use_cm', False),
        **solver_kwargs,
    )
    print(f"saved: {case['fi']}")
    print(f"grid: {Pall.shape[:2]} | indices: {case['indx'].tolist()} | additions: {case['n_add']}")
    print(f"LP settings: {solver_kwargs if solver_kwargs else 'default'}")
    print(
        f"LP curves: {len(result['numerical']['boundaries'])} | "
        f"K curves: {len(result['ana_k']['boundaries'])} | "
        f"M curves: {len(result['ana_m']['boundaries'])}"
    )
    return result


## Grid

The defaults use `Np_grid = 100`, `Nmu_grid = 100`, `NP = 14`, and `mu` up to `25`. Reduce the grid sizes for an interactive first run.

Large `mu` with small `NP` can be numerically delicate. For example, using
only `P0, ..., P9` while `mu` is near `100` puts almost all probability mass
outside the selected photon numbers. Analytical criteria may still report a
formal violation in very small probabilities, while the LP witness works with
double-precision objective coefficients and a finite decision threshold. Use
`high_precision_witness = True` as a diagnostic if you suspect the threshold
is hiding a weak witness. If the selected probability mass is extremely
small, it can be useful to compare analytical and numerical criteria. Analytical criteria use local algebraic structure and signs, but even analytical criteria at such **tiny probabilities should be treated with care**, because roundoff, truncation, and underflow can make a formal violation meaningless.


The scan also distinguishes non-physical input from nonclassicality. If a grid point contains a negative probability, a non-finite value, or selected probabilities whose sum is already larger than one, it is stored as `NaN`, skipped, and shown in white.


In [ ]:
Np_grid = 100
Nmu_grid = 100
NP = 14

p_all = np.linspace(0.0001, 0.9999, num=Np_grid, endpoint=True)
mu_all = np.linspace(0.0001, 25.0001, num=Nmu_grid, endpoint=True)
indx_all = np.arange(NP)

high_precision_witness = False   # Test with True as well
run_two_boson_qutip = True      # Set True to run the optional QuTiP example
lp_precision_kwargs = {}
if high_precision_witness:
    lp_precision_kwargs = dict(
        bias=1e-12,
        tol=1e-12,
        imtol=1e-10,
        max_iter=600,
    )

print('N_p =', p_all.shape[0], '| N_mu =', mu_all.shape[0], '| grid points =', p_all.shape[0] * mu_all.shape[0])
print('LP precision settings:', lp_precision_kwargs if lp_precision_kwargs else 'default')

## Example 1: probabilistic addition of one boson

This uses the closed-form probability helper `test_P_pba`, so it does not need
QuTiP.

In [ ]:
case_1 = dict(
    n_add=1,
    indx=indx_all,
    fi=f'../results/demo_4_pba_add1_NP{NP}.npz',
    use_ana_m=False,
)

Pall_add1 = make_pba_closed_grid(mu_all, p_all, case_1['indx'])
probability_scale_report(Pall_add1, mu_all, p_all)
data = run_pba_case(case_1, mu_all, p_all, Pall_add1)
plot_boundaries(data['numerical']['boundaries'], data['ana_k']['boundaries'], data['ana_m']['boundaries'], 'Probabilistic boson addition: one boson', mu_all, p_all)

data = load_scan_archive(case_1['fi'])
detection_summary(data, 'one boson')
plot_map(data, 'PBA map: one boson')


## Example 2: same one-boson scan with reversed y direction

The LP scan reuses a witness from the previous point along the `p` axis.
This can make the scan direction matter close to a boundary. The
cell below repeats Example 1 but visits `p` from high to low by setting
`y_scan_direction='decreasing'`.

In [ ]:
case_1_y_decreasing = dict(
     n_add=1,
     indx=indx_all,
     fi=f'../results/demo_4_pba_add1_NP{NP}_y_decreasing.npz',
     use_ana_m=False,
     solver_kwargs={**lp_precision_kwargs, 'y_scan_direction': 'decreasing'},
 )

data_y_decreasing = run_pba_case(
     case_1_y_decreasing, mu_all, p_all, Pall_add1,
 )
plot_boundaries(
     data_y_decreasing['numerical']['boundaries'], data_y_decreasing['ana_k']['boundaries'], data_y_decreasing['ana_m']['boundaries'],
     'Probabilistic boson addition: one boson, y decreasing',
     mu_all, p_all,
 )

data_y_decreasing = load_scan_archive(case_1_y_decreasing['fi'])
detection_summary(data_y_decreasing, 'one boson, y decreasing')
plot_map(data_y_decreasing, 'PBA map: one boson, y decreasing')


Consequently, it may be useful to compare increasing and decreasing scan directions, or to repeat the scan after swapping the x and y axes. Those extra consistency checks are left to the user. 

## Example 3: probabilistic addition of two bosons

This example uses the optional QuTiP-backed coherent-state construction. It is slower
than the closed-form model because a density matrix is constructed for every
grid point and every requested photon number. This can take several minutes (~10min) for
a standard single-core run. 

In [ ]:
case_2 = dict(
    n_add=2,
    indx=indx_all,
    fi=f'../results/demo_4_pba_add2_NP{NP}.npz',
    use_ana_m=False,
)

if HAS_QUTIP and run_two_boson_qutip:
    Pall_add2 = make_pba_qutip_grid(mu_all, p_all, case_2['indx'], n_add=2, ns=100)
    data = run_pba_case(case_2, mu_all, p_all, Pall_add2)
    plot_boundaries(data['numerical']['boundaries'], data['ana_k']['boundaries'], data['ana_m']['boundaries'], 'Probabilistic boson addition: two bosons', mu_all, p_all)

    data = load_scan_archive(case_2['fi'])
    detection_summary(data, 'two bosons')
    plot_map(data, 'PBA map: two bosons')
else:
    print('Skipping two-boson example: install QuTiP and enable run_two_boson_qutip to run it.')


## CSV structure for experimental or external data

A CSV file suitable for a 2D fast scan should contain one row per grid point.
The first two columns are the scan coordinates, here `mu` and `p`. The
remaining columns are photon-number probabilities whose names encode their
indices: for example `P0`, `P1`, ..., `P9` for `indx=np.arange(10)`.

For example:

```text
mu,p,P0,P1,P2,...,P9
0.0001,0.0001,0.9998,0.0001,...
...
```

The row order does not matter as long as each `(mu, p)` pair appears once.

In [ ]:
def save_pall_csv(path, mu_values, p_values, Pall, indx):
    """Save Pall[i_p, i_mu, j] as rows: mu,p,P{indx[0]},P{indx[1]},..."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    header = ','.join(['mu', 'p'] + [f'P{int(k)}' for k in indx])
    rows = []
    for ip, p in enumerate(p_values):
        for imu, mu in enumerate(mu_values):
            rows.append([mu, p, *Pall[ip, imu, :]])
    np.savetxt(path, np.asarray(rows), delimiter=',', header=header, comments='')


def load_pall_csv(path, indx):
    """Load rows mu,p,P{index} into mu_values, p_values, Pall."""
    data = np.genfromtxt(path, delimiter=',', names=True)
    data = np.atleast_1d(data)
    mu_values = np.unique(data['mu'])
    p_values = np.unique(data['p'])
    Pall = np.full((len(p_values), len(mu_values), len(indx)), np.nan)

    mu_pos = {value: i for i, value in enumerate(mu_values)}
    p_pos = {value: i for i, value in enumerate(p_values)}
    for row in data:
        ip = p_pos[row['p']]
        imu = mu_pos[row['mu']]
        for j, k in enumerate(indx):
            Pall[ip, imu, j] = row[f'P{int(k)}']

    if np.isnan(Pall).any():
        raise ValueError('CSV grid is incomplete: some (mu, p) points are missing')
    return mu_values, p_values, Pall

In [ ]:
csv_path = f'../results/demo_4_pba_add1_NP{NP}.csv'
save_pall_csv(csv_path, mu_all, p_all, Pall_add1, case_1['indx'])
mu_csv, p_csv, Pall_csv = load_pall_csv(csv_path, case_1['indx'])

print('saved CSV:', csv_path)
print('loaded shape:', Pall_csv.shape)
print('same probabilities:', np.allclose(Pall_csv, Pall_add1))

The loaded CSV grid can be analyzed directly. This is the same entry point an
experiment would use after converting measured photon-number histograms into
columns `P0`, `P1`, ... on a rectangular `(mu, p)` grid.

In [ ]:
csv_case = dict(
    n_add=1,
    indx=case_1['indx'],
    fi=f'../results/demo_4_pba_add1_NP{NP}_from_csv.npz',
    use_ana_m=False,
)

data = run_pba_case(csv_case, mu_csv, p_csv, Pall_csv)
data = load_scan_archive(csv_case['fi'])
detection_summary(data, 'one boson from CSV')
plot_boundaries(data['numerical']['boundaries'], data['ana_k']['boundaries'], data['ana_m']['boundaries'], 'PBA from CSV: one boson', mu_csv, p_csv)


## Further examples

- [demo_1_lp_solver](demo_1_lp_solver.ipynb): Single-point LP witnesses, a Nelder–Mead cross-check, and a Jaynes–Cummings example.
- [demo_2_analytical_vs_numerical](demo_2_analytical_vs_numerical.ipynb): Probability-plane scans comparing LP with analytical K and M criteria.
- [demo_3_jaynes_cummings](demo_3_jaynes_cummings.ipynb): Jaynes–Cummings maps with K/M comparisons and plotting of multiple boundary branches.
- [demo_5_counting_matrix_vs_numerics](demo_5_counting_matrix_vs_numerics.ipynb): Probability-plane comparisons of LP, K, M, and counting-matrix criteria, including nonconsecutive indices and saved-scan plotting.
- [demo_6_counting_matrix_jc](demo_6_counting_matrix_jc.ipynb): Jaynes–Cummings counting-matrix comparisons for several photon cutoffs, plus a selected-index K/M/CM comparison.
- [demo_7_vicinity_of_thermal_balance](demo_7_vicinity_of_thermal_balance.ipynb): Bisection near Jaynes–Cummings thermal balance, comparing K, counting matrices, and LP; cutoff comparisons and logarithmic-distance plots.
